La prima domanda di ricerca a cui cerchiamo di dare una risposta è: il vinile viene acquistato principalmente per l'ascolto o come oggetto di status symbol? L'obiettivo è comprendere in quali casi il vinile viene acquistato come supporto per ascoltare un album che si apprezza particolarmente e quando, invece, diventa un oggetto decorativo o collezionistico.

Per rispondere a questa domanda l'intenzione è affidarsi a tre database:
- **Billboard Vinyl Albums** (2025), che permette di valutare quali sono gli album più venduti in formato fisico nel mercato americano
- **Spotify Charts**, che permette di valutare quali sono gli album più ascoltati sulle piattaforme di streaming
- **Official Charts (UK)**, che permette di confrontare i risultati con un mercato simile dal punto di vista linguistico e dell'offerta musicale

Il confronto tra le classifiche di vendita fisiche e quelle di streaming è il cuore metodologico di questa analisi: laddove un album vende molto in formato vinile ma registra scarsi ascolti in streaming, è ragionevole ipotizzare che l'acquisto sia motivato da ragioni che vanno oltre il semplice ascolto, come il collezionismo, il valore estetico dell'oggetto o l'identificazione con un certo gusto culturale.

**Primo ostacolo nella ricerca dei dati**: con l'appoggio di Claude.ai nella raccolta di [Billboard Vinyl Album](https://www.billboard.com/charts/year-end/) sono stati tentati tre approcci di web scraping, nessuno dei quali ha prodotto risultati utilizzabili.
- Il primo tentativo ha utilizzato le librerie requests e BeautifulSoup, ma si è rivelato inefficace perché il sito di Billboard carica i contenuti dinamicamente tramite JavaScript.
- Il secondo e il terzo tentativo hanno utilizzato Selenium e successivamente Playwright, due strumenti in grado di simulare un browser reale ed eseguire JavaScript.
Nonostante i tentativi di risolvere il problema aggiornando le dipendenze e modificando la configurazione, l'errore ha persistito.

La causa principale del fallimento di tutti e tre gli approcci è riconducibile a due fattori: la natura dinamica del sito di Billboard, che richiede l'esecuzione di JavaScript per caricare i contenuti, e le limitazioni dell'ambiente Google Colab, che rende difficile la gestione di browser headless.


A seguito delle problematiche riscontrare con lo scraping dei dati di Billboard, si è scelto di utilizzare **Discogs**, un database aperto e liberalmente interrogabile tramite API. Al contrario di Billboard, Discogs mette a disposizone un'API REST gratuita e ben documentata, che consente di interrogare il database direttamente tramite codice Python senza incorrere in problemi tecnici e legali. Tramite l'utilizzo di Claude.ai è stato possibile generare il codice necessario per interrogare l'API di Discogs e raccogliere i dati relativi ai vinili più popolari nel mercato americano nel 2025.


Il codice scarica automaticamente i dati suddivisi in più pagine di risultati e li organizza in un dataset strutturato contenente le seguenti informazioni per ciascun album: titolo, anno di pubblicazione, genere musicale, stile e due indici di popolarità forniti dalla comunità di Discogs:
-  il numero di utenti che dichiarano di possedere l'album (campo "hanno")
- il numero di utenti che lo desiderano (campo "vogliono").

Questi ultimi due valori sono particolarmente rilevanti per rispondere alla prima domanda di ricerca: un album con un alto rapporto tra "vogliono" e "hanno" suggerisce infatti che venga percepito come un oggetto desiderabile al di là del semplice ascolto, avvicinandosi alla categoria dello status symbol.

I dati raccolti vengono salvati automaticamente in un file CSV e visualizzati tramite una tabella interattiva che permette di filtrare i risultati per anno, genere o titolo.

In [ ]:
!pip install requests pandas ipywidgets

import requests
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
from collections import defaultdict
import time

# ── CONFIGURAZIONE ────────────────────────────────────────────────────
TOKEN = "YmYWIbBkQVihZyWWxXIAPhvMZjSeWeRGSxPGOZnx"

headers = {
    "Authorization": f"Discogs token={TOKEN}",
    "User-Agent": "ProgettoVinili/1.0"
}

# ── RACCOLTA DATI DA DISCOGS ──────────────────────────────────────────
print("🔍 Recupero dati da Discogs...")

dati = []
for pagina in range(1, 6):
    print(f"  Scarico pagina {pagina}/5...")
    url = "https://api.discogs.com/database/search"
    params = {
        "type": "release",
        "format": "Vinyl",
        "country": "US",
        "year": 2025,
        "sort": "have",
        "sort_order": "desc",
        "per_page": 50,
        "page": pagina
    }
    response = requests.get(url, headers=headers, params=params)
    risultati = response.json().get("results", [])

    for r in risultati:
        dati.append({
            "titolo": r.get("title", "N/A"),
            "anno": r.get("year", "N/A"),
            "genere": ", ".join(r.get("genre", [])),
            "stile": ", ".join(r.get("style", [])),
            "etichetta": ", ".join(r.get("label", [])),
            "hanno": r.get("community", {}).get("have", 0),
            "vogliono": r.get("community", {}).get("want", 0),
        })
    time.sleep(1)

df = pd.DataFrame(dati)
df = df.sort_values("hanno", ascending=False).reset_index(drop=True)
print(f"✅ Trovati {len(df)} vinili.\n")

# Salva in CSV
df.to_csv("discogs_vinili_2025.csv", index=False)

# ── TABELLA COMPLETA ──────────────────────────────────────────────────
def genera_tabella(dataframe, titolo=""):
    righe_html = ""
    colore_alt = False
    for _, r in dataframe.iterrows():
        bg = "#f9f9f9" if colore_alt else "#ffffff"
        colore_alt = not colore_alt
        righe_html += (
            f"<tr style='background:{bg}'>"
            f"<td style='padding:5px 12px; text-align:center'><b>{r['anno']}</b></td>"
            f"<td style='padding:5px 12px'>{r['titolo']}</td>"
            f"<td style='padding:5px 12px'>{r['genere']}</td>"
            f"<td style='padding:5px 12px'>{r['stile']}</td>"
            f"<td style='padding:5px 12px; text-align:center'>{r['hanno']}</td>"
            f"<td style='padding:5px 12px; text-align:center'>{r['vogliono']}</td>"
            f"</tr>"
        )
    return HTML(f"""
    <h3>🎵 {titolo}</h3>
    <p style='color:gray'>Totale: <b>{len(dataframe)}</b> vinili</p>
    <div style='max-height:500px; overflow-y:auto'>
    <table border='1' cellspacing='0' style='border-collapse:collapse; font-size:13px; width:100%'>
      <thead style='background:#1a1a2e; color:white; position:sticky; top:0'>
        <tr>
          <th style='padding:7px 12px'>Anno</th>
          <th style='padding:7px 12px'>Titolo</th>
          <th style='padding:7px 12px'>Genere</th>
          <th style='padding:7px 12px'>Stile</th>
          <th style='padding:7px 12px'>Hanno</th>
          <th style='padding:7px 12px'>Vogliono</th>
        </tr>
      </thead>
      <tbody>{righe_html}</tbody>
    </table>
    </div>
    """)

display(genera_tabella(df, "Vinili più popolari su Discogs – USA 2025"))

# ── CONTEGGIO PER GENERE ──────────────────────────────────────────────
conteggio_genere = defaultdict(int)
for genere in df["genere"]:
    for g in genere.split(", "):
        if g and g != "N/A":
            conteggio_genere[g] += 1

righe_genere = "".join(
    f"<tr><td style='padding:4px 16px'><b>{g}</b></td>"
    f"<td style='padding:4px 16px; text-align:center'>{'🎵' * min(n, 10)} {n}</td></tr>"
    for g, n in sorted(conteggio_genere.items(), key=lambda x: -x[1])
)

display(HTML(f"""
<h3>📊 Distribuzione per genere</h3>
<table border='1' cellspacing='0' style='border-collapse:collapse; font-size:13px'>
  <thead style='background:#1a1a2e; color:white'>
    <tr>
      <th style='padding:7px 16px'>Genere</th>
      <th style='padding:7px 16px'>N° vinili</th>
    </tr>
  </thead>
  <tbody>{righe_genere}</tbody>
</table>
"""))

# ── RICERCA INTERATTIVA ───────────────────────────────────────────────
testo_input = widgets.Text(
    value="Rock",
    description="🔎 Cerca:",
    placeholder="Anno (es. 2025), genere (es. Rock) o titolo",
    layout=widgets.Layout(width="420px")
)
btn = widgets.Button(description="Cerca", button_style="primary", icon="search")
output = widgets.Output()

def cerca(b):
    output.clear_output()
    query = testo_input.value.strip()
    with output:
        if not query:
            print("⚠️ Inserisci un anno, un genere o un titolo.")
            return

        if query.isdigit():
            risultati = df[df["anno"] == query]
        else:
            q = query.lower()
            risultati = df[
                df["titolo"].str.lower().str.contains(q, na=False) |
                df["genere"].str.lower().str.contains(q, na=False) |
                df["stile"].str.lower().str.contains(q, na=False)
            ]

        if risultati.empty:
            display(HTML(f"<span style='color:red'>❌ Nessun risultato per '<b>{query}</b>'.</span>"))
            return

        display(genera_tabella(risultati, f"Risultati per \"{query}\""))

btn.on_click(cerca)
display(HTML("<h3>🔎 Filtra per anno, genere o titolo</h3>"))
display(widgets.HBox([testo_input, btn]), output)

I dati raccolti permettono di affermare che il genere più rappresentato tra i vinili più venduti negli Stati Uniti nel 2025 è il **pop**, al quale si avvicina solo l'hip-hop. A partire dal terzo posto, la presenza di ciascun genere nella classifica scende drasticamente — in alcuni casi fino a un solo titolo.

Un dato particolarmente interessante riguarda il rapporto tra il numero di utenti che possiedono un vinile ("hanno") e quelli che lo desiderano ("vogliono"). Per i generi più rappresentati, come pop e rock, il numero di possessori supera in modo spesso molto netto quello dei potenziali acquirenti: questo suggerisce che questi album vengano effettivamente acquistati e non rimangano semplicemente oggetto di desiderio. Lo stesso schema si ripete anche per i generi meno rappresentati (reggae, latin e musica per bambini) dove il rapporto tra possessori e desideranti risulta analogamente sbilanciato a favore dei primi.

Questo dato, trasversale a tutti i generi analizzati, sembra indicare che il vinile venga acquistato principalmente per essere posseduto fisicamente — che si tratti di ascolto o di collezione — piuttosto che restare un oggetto del desiderio inaccessibile. Questa osservazione contribuisce a rispondere alla prima domanda di ricerca, suggerendo che il fenomeno del vinile come puro status symbol sia meno diffuso di quanto si potrebbe ipotizzare.

Per la comparazione con i dati del **mercato musicale britannico**, tramite Claude è stato creato un file CVS con i primi dieci album della classifica riguardante i vinili di Official Charts.

In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
df_uk = pd.read_csv("official_charts_uk_vinyl_2025.csv")
print(df_uk)

Confrontando le due raccolte di dati, è possibile rintracciare alcune caratteristiche comuni
- la maggior parte degli album e degli artisti presenti nella classifica britannica si rintracciano anche in quella americana, probabilmente per una questione di somiglianza linguistica. Eccezione fatta per i progetti di alcuni artisti inglesi, come per esempio gli Oasis, che sono presenti solo nella lista tratta dal portale britannico. L'unione di questi due dati potrebbe sottolineare come la musica statunitense riesce a superare facilmente i confini del proprio paese, mentre quella britannica trova più difficoltà nella diffusione oltre oceano - mantenendo tuttavia una fetta importante del mercato interno.
- anche nella classifica britannica i generi che trovano maggiore spazio sono il pop e il rock; quindi è possibile affermare che in entrambi i mercati questi due generi musicali sono quelli che riescono a generare maggiore interesse nell'acquisto di vinili.

Per completare questa prima fase di analisi, è necessario analizzare se i prodotti fisici che vengono acquistati in quantità maggiore rappresentano i gusti rintracciabili nello streaming. Qualora ci fosse una congruenza tra questi due dati, si potrebbe pensare che l'acquisto è dettato da una questione di gusto e apprezzamento personale; diversamente, si potrebbe pensare che il vinile rimanga, seppur in una certa dimensione, un oggetto capace di trasmettere un valore sociale e di appartenza a un determinato gruppo.

**Secondo ostacolo nella ricerca dati**: a causa di una questione legale legata ai termini di accettazione di SpotifyCharts, non è possibile utilizzare il loro database tramite scraping o utilizzo di qualsiasi forma di ricerca automatica. Per questo si è deciso di ricorrere, in prima battuta, a **Kworb.net**, un sito che accorpa i dati di ascolto sia di Spotify sia di Apple Music.  Tuttavia, Kworb non dispone di una sezione dedicata agli album — fornisce esclusivamente dati sui singoli brani — e non era quindi perfettamente adatto a rispondere alla domanda di ricerca, che richiede un confronto tra album venduti in formato fisico e album più ascoltati in streaming. Si è quindi optato per **Last.fm** come fonte alternativa.

In [ ]:
import requests
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
import time

# ── CONFIGURAZIONE ────────────────────────────────────────────────────
API_KEY = "a9af19a8638c6a434d9c6a8dc6892262"
BASE_URL = "https://ws.audioscrobbler.com/2.0/"

# ── RACCOLTA DATI ─────────────────────────────────────────────────────
print("🔍 Recupero album più ascoltati da Last.fm...")

dati = []
pagine = 5

for pagina in range(1, pagine + 1):
    print(f"  Scarico pagina {pagina}/{pagine}...")
    params = {
        "method": "tag.gettopalbums",
        "tag": "all",
        "api_key": API_KEY,
        "format": "json",
        "limit": 50,
        "page": pagina
    }

    response = requests.get(BASE_URL, params=params)
    print(f"  Stato risposta pagina {pagina}:", response.status_code)

    risultati = response.json().get("albums", {}).get("album", [])
    print(f"  Album trovati pagina {pagina}:", len(risultati))

    for album in risultati:
        dati.append({
            "posizione": len(dati) + 1,
            "titolo": album.get("name", "N/A"),
            "artista": album.get("artist", {}).get("name", "N/A"),
            "ascolti": album.get("playcount", 0),
            "ascoltatori": album.get("listeners", 0)
        })

    time.sleep(0.5)

df = pd.DataFrame(dati)
print(f"\n✅ Trovati {len(df)} album.")
print(df.head())

Come si evince dal risultato della procedura precedente, i dati ottenuti tramite lo scraping del sito last.fm non sono ottimali al fine della comparazione tra streaming e vendite.